# **Analyse thématique des avis clients**

**Objectif** : Identifier les thèmes principaux et sentiments évoqués dans les avis (propreté, service, localisation, etc.)

**Ce notebook utilise** :
- Les données pré-traitées : `data/processed/avis_preprocesses.parquet`
- Les tokens nettoyés (`tokens_clean`) pour l'analyse

**Étapes** :
- Chargement des données pré-traitées
- Exploration des thèmes par topic modeling (LDA)
- Analyse des sentiments (positif/négatif/neutre)
- Analyse comparative français/anglais (optionnel)

## **Imports**

In [1]:
# imports pour l'analyse thématique et des sentiments
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from textblob import TextBlob
from pathlib import Path
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
import warnings
warnings.filterwarnings('ignore')

In [13]:
%pip install textblob-fr

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.2/561.2 kB 17.6 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.


In [22]:
import importlib

import utils
importlib.reload(utils.sentiment_analysis)

<module 'utils.sentiment_analysis' from '/workspaces/assistant-rag/src/utils/sentiment_analysis.py'>

In [23]:
# custom imports
from utils.sentiment_analysis import analyser_sentiment

In [2]:
# configuration
PROJET_ROOT = Path.cwd().parent.parent
DATA_PROCESSED = PROJET_ROOT / "data" / "processed"

pd.set_option('display.max_colwidth', 200)

## **Chargement des données**

In [3]:
# chargement des données pré-traitées
df_avis = pd.read_parquet(DATA_PROCESSED / "avis_preprocesses.parquet")
print(f"Données chargées : {df_avis.shape}")
df_avis[['avis_nettoye', 'langue', 'tokens_clean']].head()

Données chargées : (5946, 9)


,avis_nettoye,langue,tokens_clean
0,Chambre confortable mais décoration un peu démodée. Le petit-déjeuner est bon mais sans originalité particulière.,fr,"[Chambre, confortable, décoration, démodée, petit-déjeuner, bon, originalité, particulière]"
1,Le spa propose un traitement signature exclusif développé avec un aromathérapeute de renommée mondiale spécialement pour Hotel De La Promenade.,fr,"[spa, propose, traitement, signature, exclusif, développé, aromathérapeute, renommée, mondiale, spécialement, Hotel, Promenade]"
2,Un séjour correct mais qui ne justifie pas pleinement son classement luxe. Service professionnel mais sans cette touche personnelle qui fait la différence.,fr,"[séjour, correct, justifie, pleinement, classement, luxe, Service, professionnel, touche, personnelle, différence]"
3,The laundry service express saved our gala dinner following a last-minute incident.,en,"[laundry, service, express, saved, gala, dinner, following, -, minute, incident]"
4,L'exposition de collection d'art contemporain de l'hôtel est présentée lors de visites privées par un historien d'art passionné qui en révèle toutes les subtilités.,fr,"[exposition, collection, art, contemporain, hôtel, présentée, visites, privées, historien, art, passionné, révèle, subtilités]"


## **Analyse thématique**

### **1. Préparation des données**

In [4]:
# reconstitution du texte à partir des tokens pour LDA
df_avis['texte_pour_lda'] = df_avis['tokens_clean'].apply(lambda x: ' '.join(x))
df_avis[['langue', 'texte_pour_lda']].head()

,langue,texte_pour_lda
0,fr,Chambre confortable décoration démodée petit-déjeuner bon originalité particulière
1,fr,spa propose traitement signature exclusif développé aromathérapeute renommée mondiale spécialement Hotel Promenade
2,fr,séjour correct justifie pleinement classement luxe Service professionnel touche personnelle différence
3,en,laundry service express saved gala dinner following - minute incident
4,fr,exposition collection art contemporain hôtel présentée visites privées historien art passionné révèle subtilités


### **2. Vectorisation**

In [5]:
# création de la matrice documents-termes
vectorizer = CountVectorizer(max_features=1000)
X = vectorizer.fit_transform(df_avis['texte_pour_lda'])
print(f"matrice shape : {X.shape}")

matrice shape : (5946, 1000)


### **3. Modélisation LDA**

In [6]:
# entraînement du modèle LDA
lda = LatentDirichletAllocation(n_components=5, random_state=42, n_jobs=-1)
lda.fit(X)

,"n_components n_components: int, default=10Number of topics... versionchanged:: 0.19 ``n_topics`` was renamed to ``n_components``",5
,"doc_topic_prior doc_topic_prior: float, default=NonePrior of document topic distribution `theta`. If the value is None,defaults to `1 / n_components`.In [1]_, this is called `alpha`.",None
,"topic_word_prior topic_word_prior: float, default=NonePrior of topic word distribution `beta`. If the value is None, defaultsto `1 / n_components`.In [1]_, this is called `eta`.",None
,"learning_method learning_method: {'batch', 'online'}, default='batch'Method used to update `_component`. Only used in :meth:`fit` method.In general, if the data size is large, the online update will be muchfaster than the batch update.Valid options:- 'batch': Batch variational Bayes method. Use all training data in each EM update. Old `components_` will be overwritten in each iteration.- 'online': Online variational Bayes method. In each EM update, use mini-batch of training data to update the ``components_`` variable incrementally. The learning rate is controlled by the ``learning_decay`` and the ``learning_offset`` parameters... versionchanged:: 0.20 The default learning method is now ``""batch""``.",'batch'
,"learning_decay learning_decay: float, default=0.7It is a parameter that control learning rate in the online learningmethod. The value should be set between (0.5, 1.0] to guaranteeasymptotic convergence. When the value is 0.0 and batch_size is``n_samples``, the update method is same as batch learning. In theliterature, this is called kappa.",0.7
,"learning_offset learning_offset: float, default=10.0A (positive) parameter that downweights early iterations in onlinelearning. It should be greater than 1.0. In the literature, this iscalled tau_0.",10.0
,"max_iter max_iter: int, default=10The maximum number of passes over the training data (aka epochs).It only impacts the behavior in the :meth:`fit` method, and not the:meth:`partial_fit` method.",10
,"batch_size batch_size: int, default=128Number of documents to use in each EM iteration. Only used in onlinelearning.",128
,"evaluate_every evaluate_every: int, default=-1How often to evaluate perplexity. Only used in `fit` method.set it to 0 or negative number to not evaluate perplexity intraining at all. Evaluating perplexity can help you check convergencein training process, but it will also increase total training time.Evaluating perplexity in every iteration might increase training timeup to two-fold.",-1
,"total_samples total_samples: int, default=1e6Total number of documents. Only used in the :meth:`partial_fit` method.",1000000.0
,"perp_tol perp_tol: float, default=1e-1Perplexity tolerance. Only used when ``evaluate_every`` is greater than 0.",0.1


### **4. Résultats**

In [7]:
# affichage des mots-clés par topic/thème
mots = vectorizer.get_feature_names_out()

for topic_idx, topic in enumerate(lda.components_):
    mots_importants = [mots[i] for i in topic.argsort()[:-10-1:-1]]
    print(f"Topic {topic_idx}: {', '.join(mots_importants)}")

Topic 0: hotel, room, check, told, lobby, desk, nice, stay, promenade, la
Topic 1: room, hotel, bathroom, good, night, rooms, small, bed, service, stay
Topic 2: hotel, room, ottawa, great, location, parliament, staff, promenade, la, de
Topic 3: hôtel, chambre, bien, hotel, promenade, service, ottawa, chambres, personnel, petit
Topic 4: hotel, promenade, la, de, ottawa, stay, room, service, staff, stayed


**Observations:**
- **`de`** et **`la`** : spaCy ne les a pas lemmatisés car ce sont des déterminants/prépositions, mais ils auraient dû être filtrés par les stop words, à moins que le vocabulaire stops words est limité. <br />
Problème : soit ils ne sont pas dans la liste de stop words de spaCy, soit le filtrage n'a pas fonctionné pour ces cas (peu probables).
- **`hotel`** vs **`hôtel`** : On voit les deux formes, ce qui suggère que la lemmatisation n'a pas unifié les variantes.

In [29]:
# distribution des topics pour chaque document
topic_distribution = lda.transform(X)
print(f"shape de topic_distribution : {topic_distribution.shape}")

shape de topic_distribution : (5946, 5)


In [30]:
# interprétation métier des topics
noms_topics = {
    0: "Service / Accueil",      # check, told, lobby, desk, staff
    1: "Confort / Chambres",     # bathroom, rooms, small, bed
    2: "Localisation",           # ottawa, great, location, parliament
    3: "Service en français",    # hôtel, chambre, personnel, service
    4: "Expérience globale"      # stay, stayed, service, staff
}

# fusion des topics 3 et 4 si nécessaire
# ou on peut garder 3 comme "Service FR" et 4 comme "Expérience EN"

df_avis['topic_dominant'] = topic_distribution.argmax(axis=1)
df_avis['topic_nom'] = df_avis['topic_dominant'].map(noms_topics)

# distribution
df_avis['topic_nom'].value_counts()

topic_nom
Expérience globale     1877
Localisation           1762
Confort / Chambres     1014
Service en français     832
Service / Accueil       461
Name: count, dtype: int64

In [31]:
# essai avec plus de topics pour mieux distinguer les thèmes
lda_8 = LatentDirichletAllocation(n_components=8, random_state=42, n_jobs=-1)
lda_8.fit(X)

mots = vectorizer.get_feature_names_out()
for topic_idx, topic in enumerate(lda_8.components_):
    mots_importants = [mots[i] for i in topic.argsort()[:-8-1:-1]]
    print(f"Topic {topic_idx}: {', '.join(mots_importants)}")

Topic 0: dated, lobby, hotel, soon, nice, walked, impression, spa
Topic 1: hotel, room, rooms, good, de, la, promenade, bathroom
Topic 2: hotel, room, ottawa, great, parliament, location, good, staff
Topic 3: hôtel, chambre, bien, hotel, promenade, ottawa, chambres, personnel
Topic 4: service, suite, hotel, promenade, hilton, la, de, diamond
Topic 5: hotel, promenade, la, de, ottawa, stay, staff, room
Topic 6: room, hotel, check, promenade, la, de, stay, night
Topic 7: service, bar, room, mini, asked, night, took, hour


In [32]:
# utilisation du modèle à 8 topics
lda_final = LatentDirichletAllocation(n_components=8, random_state=42, n_jobs=-1)
lda_final.fit(X)

# distribution des topics pour chaque document
topic_distribution = lda_final.transform(X)
df_avis['topic_dominant'] = topic_distribution.argmax(axis=1)
df_avis['topic_proba'] = topic_distribution.max(axis=1)

# noms métier pour les 8 topics
noms_topics = {
    0: "Décoration / Vétusté",
    1: "Chambres (confort)",
    2: "Localisation",
    3: "Expérience francophone",
    4: "Service / Fidélité",
    5: "Expérience générale",
    6: "Check-in / Attente",
    7: "Service / Problèmes"
}

df_avis['topic_nom'] = df_avis['topic_dominant'].map(noms_topics)

# distribution
df_avis['topic_nom'].value_counts()

topic_nom
Expérience générale       1461
Localisation              1449
Chambres (confort)        1099
Check-in / Attente         828
Expérience francophone     708
Service / Fidélité         251
Décoration / Vétusté        95
Service / Problèmes         55
Name: count, dtype: int64

## **Analyse des sentiments**

In [24]:
df_avis['sentiment_score'] = df_avis.apply(
    lambda row: analyser_sentiment(row['avis_nettoye'], row['langue']), 
    axis=1
)
df_avis['sentiment_categorie'] = pd.cut(
    df_avis['sentiment_score'], 
    bins=[-1, -0.1, 0.1, 1], 
    labels=['negatif', 'neutre', 'positif']
)

In [10]:
df_avis[['avis_nettoye', 'sentiment_score', 'sentiment_categorie']].head()

,avis_nettoye,sentiment_score,sentiment_categorie
0,Chambre confortable mais décoration un peu démodée. Le petit-déjeuner est bon mais sans originalité particulière.,0.0,neutre
1,Le spa propose un traitement signature exclusif développé avec un aromathérapeute de renommée mondiale spécialement pour Hotel De La Promenade.,0.0,neutre
2,Un séjour correct mais qui ne justifie pas pleinement son classement luxe. Service professionnel mais sans cette touche personnelle qui fait la différence.,0.0,neutre
3,The laundry service express saved our gala dinner following a last-minute incident.,0.0,neutre
4,L'exposition de collection d'art contemporain de l'hôtel est présentée lors de visites privées par un historien d'art passionné qui en révèle toutes les subtilités.,0.0,neutre


In [11]:
# distribution des sentiments
df_avis['sentiment_categorie'].value_counts()

sentiment_categorie
positif    4262
neutre     1530
negatif     150
Name: count, dtype: int64

**Les scores à 0.0 pour les premiers échantillons semblent suspects. TextBlob n'a probablement pas bien traité le français.**

In [12]:
# vérification sur un échantillon anglais
echantillon_en = df_avis[df_avis['langue'] == 'en'].sample(5)
echantillon_en['avis_nettoye'].apply(analyser_sentiment)

43      0.666667
4951    0.086894
2550    0.534782
2929   -0.075000
4033    0.484630
Name: avis_nettoye, dtype: float64

In [25]:
# test direct sur un échantillon français
# test de la fonction sur des exemples

exemples = [
    ("Ce service est absolument exceptionnel, je recommande!", "fr"),
    ("Service médiocre, je suis déçu.", "fr"),
    ("Amazing experience, will come back!", "en"),
    ("Terrible experience, never again.", "en"),
]

for texte, langue in exemples:
    score = analyser_sentiment(texte, langue) # calcul du score de sentiment
    print(f"{langue}: {score:.3f} - {texte[:50]}...")

fr: 0.325 - Ce service est absolument exceptionnel, je recomma...
fr: -0.700 - Service médiocre, je suis déçu....
en: 0.300 - Amazing experience, will come back!...
en: -1.000 - Terrible experience, never again....


**La fonction fonctionne correctement. Les premiers avis étaient donc vraiment neutres.**

In [26]:
# statistiques des scores de sentiment
df_avis['sentiment_score'].describe()

count    5946.000000
mean        0.229918
std         0.180452
min        -1.000000
25%         0.123523
50%         0.231049
75%         0.332700
max         1.000000
Name: sentiment_score, dtype: float64

## **Distribution des sentiments**

In [27]:
# distribution des sentiments par langue
df_avis.groupby('langue')['sentiment_categorie'].value_counts()

langue  sentiment_categorie
en      positif                4109
        neutre                  797
        negatif                 128
fr      positif                 603
        neutre                  249
        negatif                  58
it      neutre                    1
        positif                   0
        negatif                   0
Name: count, dtype: int64

In [33]:
# sauvegarde complète
df_avis.to_parquet(DATA_PROCESSED / "analyse_complete.parquet")
print("données sauvegardées")

données sauvegardées


## **Conclusions**

- **Analyse thématique** : 8 topics identifiés couvrant décoration, chambres, localisation, service, et expérience client. Les avis français et anglais forment des clusters distincts, confirmant la pertinence du traitement bilingue.
- **Sentiments** : 79% positifs, 18% neutres, 3% négatifs. Les avis français sont plus critiques (66% positifs) que les anglais (82% positifs).
- **Défis techniques** : Persistance de stop words ("de", "la") dans les topics malgré le filtrage. Adaptation du modèle de sentiment au français avec textblob-fr.